# Dimensionality Reduction — PCA, t-SNE, UMAP, and Autoencoders

**Author:** Shivani Bokka  
**Dataset:** MNIST Digits (sklearn `load_digits` — 8×8 images, 64 features)  
**Goal:** Understand when and why to reduce dimensions — and which method to use

---

## What Is This Notebook About?

High-dimensional data is everywhere in machine learning — images, text embeddings, sensor arrays, genomic data. But working directly with hundreds or thousands of features creates real problems: models train slowly, visualizations become impossible, and irrelevant features introduce noise that hurts accuracy.

**Dimensionality reduction** is the practice of compressing data from many dimensions into fewer dimensions while retaining as much meaningful structure as possible.

This notebook covers **four major techniques** — all applied to the same MNIST digits dataset so you can directly compare their outputs:

| # | Method | Core Idea |
|---|--------|-----------|
| 1 | **PCA** | Find the directions of maximum variance; project onto them |
| 2 | **t-SNE** | Convert pairwise distances to probabilities; preserve local neighborhoods |
| 3 | **UMAP** | Preserve both local and global structure; much faster than t-SNE |
| 4 | **Autoencoder** | Train a neural network to reconstruct its input through a bottleneck |

By the end, you will know **which method to reach for** in any situation.

---

## Section 1 — Imports and Setup

We load every library we need upfront. Key packages:

- **numpy / pandas** — data wrangling
- **matplotlib / seaborn** — visualization
- **sklearn** — PCA, IncrementalPCA, t-SNE, classifiers, datasets
- **umap-learn** — UMAP (install with `pip install umap-learn`)
- **torch / torch.nn** — PyTorch autoencoder
- **time** — benchmarking speed
- **warnings** — suppress non-critical warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn — dimensionality reduction
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.datasets import load_digits

# UMAP  (pip install umap-learn if needed)
try:
    import umap
    UMAP_AVAILABLE = True
    print("UMAP is available.")
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP not installed. Run: pip install umap-learn")

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

print("All libraries imported successfully!")

---

## Section 2 — Why Reduce Dimensions?

### The Curse of Dimensionality

Imagine you have a dataset with 1,000 features per sample. At first glance, more features sound like more information — and more information sounds like a good thing.

But here is the problem: **as the number of dimensions grows, data becomes increasingly sparse.**

Think of it this way:

> In 1D, you need 10 points to have "one point every unit" across a range of 10.  
> In 2D, you need 100 points.  
> In 3D, you need 1,000 points.  
> In 1,000D, you'd need 10^1000 points to maintain the same density.  

This sparsity causes several real problems:

- **Distance metrics break down** — in very high dimensions, all points are roughly the same distance from each other. The concept of "nearest neighbor" loses meaning.
- **Models overfit** — with more features than samples, a model can memorize the training data perfectly while generalizing terribly to new data.
- **Computation explodes** — many algorithms scale quadratically or worse with the number of features.
- **Visualization is impossible** — humans can only directly see 2D or 3D plots.

---

### 4 Real Use Cases for Dimensionality Reduction

| Use Case | What It Means | Example |
|----------|---------------|----------|
| **Visualization** | Project high-D data down to 2D or 3D for plotting | Visualize which digit classes cluster together |
| **Noise Removal** | Keep only the directions with real signal; discard noise dimensions | Remove sensor noise before anomaly detection |
| **Compression** | Represent data more compactly without much information loss | Compress images for storage or transmission |
| **Preprocessing before ML** | Reduce features before feeding to a classifier | Speed up Logistic Regression on image data |

---

### When NOT to Reduce Dimensions

Dimensionality reduction is not always the right move. Avoid it when:

- **You have very few features** (e.g., 5–20 columns) — no curse to fight, reduction may just lose information
- **Interpretability matters** — PCA components are linear combinations of original features and are hard to explain to non-technical stakeholders
- **Every feature has a clear, known meaning** — a doctor looking at lab results doesn't want features combined into uninterpretable principal components
- **Your model is already fast and accurate** — don't introduce complexity without a clear payoff

---

### Load the MNIST Digits Dataset

We use sklearn's `load_digits`, which contains 1,797 samples of handwritten digits (0–9), each represented as an 8×8 pixel image (64 features). It's lightweight and runs fast — perfect for comparing all four methods.

In [ ]:
# Load the MNIST digits dataset (8x8 = 64 features)
digits = load_digits()
X = digits.data           # shape: (1797, 64)
y = digits.target         # shape: (1797,) — digit labels 0-9

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))} (digits 0–9)")
print(f"Feature range: min={X.min():.1f}, max={X.max():.1f}")
print(f"\nClass distribution:")
for digit in range(10):
    print(f"  Digit {digit}: {(y == digit).sum()} samples")

# Scale features — critical for PCA and distance-based methods
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"\nScaled X: mean ≈ {X_scaled.mean():.4f}, std ≈ {X_scaled.std():.4f}")

In [ ]:
# Show a 4x4 grid of sample digit images
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
sample_indices = np.random.RandomState(42).choice(len(X), size=16, replace=False)

for ax, idx in zip(axes.ravel(), sample_indices):
    ax.imshow(digits.images[idx], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Label: {y[idx]}', fontsize=10)
    ax.axis('off')

plt.suptitle('MNIST Digits — Sample Images (8×8 pixels, 64 features each)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: MNIST Sample Images

This is a **4×4 grid of handwritten digit images** drawn from the dataset.

- **Each cell** shows one 8×8 pixel image. The pixel values range from 0 (white) to 16 (black).
- **The title of each cell** shows the true label — the digit that was written.
- At 8×8 resolution, the images are quite small and blurry compared to the full MNIST dataset (which uses 28×28 images). You can still recognize most digits by their shape.
- **Each image = 64 numbers** (8 rows × 8 columns of pixel values). That is the raw input vector our dimensionality reduction methods will compress.

**Why this matters for dimensionality reduction:**
- These 64 pixel values are not all equally informative. Corner pixels are often zero for most digits. The signal lives in a much smaller subspace.
- The goal of dimensionality reduction is to find that smaller subspace and represent each image with just 2–20 numbers instead of 64, without losing the ability to tell the digits apart.

---

## Section 3 — PCA (Principal Component Analysis)

**PCA finds the directions of maximum variance in your data and projects everything onto those directions.**

Here is the intuition step by step:

1. **Center the data** — subtract the mean so everything is centered at the origin.
2. **Find the direction of most spread (variance)** — this is the first principal component. Imagine rotating a camera around a cloud of points until you see the widest spread. That angle is PC1.
3. **Find the next direction** — perpendicular to PC1, find the direction with the second-most variance. That is PC2.
4. **Repeat** — keep finding orthogonal directions in order of decreasing variance.
5. **Project** — represent each data point by its coordinates along the top k principal components.

### The Math (Plain English)

- **Eigenvectors** = the principal components — the directions we project onto
- **Eigenvalues** = how much variance each direction explains
- **Explained variance ratio** = eigenvalue / total variance = the fraction of information captured by each component

You do not need to memorize the math. Just know: PCA finds the most informative directions in your data and lets you keep only those.

### Key Properties of PCA
- **Linear** — PCA can only find linear combinations of features
- **Global structure preserved** — similar points in the original space tend to stay similar after PCA
- **Reversible** — you can approximately reconstruct the original data from PCA components
- **Scale sensitive** — always standardize (zero mean, unit variance) before applying PCA

In [ ]:
# Fit PCA with all 64 components to see the full variance spectrum
pca_full = PCA(n_components=64, random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# How many components to reach 95% explained variance?
n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"Components needed for 95% explained variance: {n_95}")
print(f"First 10 components explain: {cumulative_var[9]:.1%} of variance")
print(f"First 20 components explain: {cumulative_var[19]:.1%} of variance")

# Plot explained variance — bar chart + cumulative curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: first 20 components
axes[0].bar(range(1, 21), explained_var[:20], color='steelblue', edgecolor='navy', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance per Component\n(First 20 Components)')
axes[0].set_xticks(range(1, 21))

# Cumulative curve
axes[1].plot(range(1, 65), cumulative_var, color='steelblue', linewidth=2, marker='o', markersize=3)
axes[1].axhline(0.95, color='red', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].axvline(n_95, color='tomato', linestyle=':', linewidth=1.5,
                label=f'{n_95} components → 95%')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance\nvs. Number of Components')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.suptitle('PCA — Variance Analysis on MNIST Digits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Explained Variance

This is a two-panel chart that tells you how many PCA components you actually need.

**Left panel — Explained Variance per Component (bar chart):**
- **Each bar** represents one principal component, numbered 1 through 20.
- **Bar height** = the fraction of total variance that component captures. For example, a bar at 0.12 means that component captures 12% of all the information in the dataset.
- **The bars drop sharply** — the first few components carry the most information, and each additional component carries less and less. This is typical of PCA: a small number of components dominate.

**Right panel — Cumulative Explained Variance (line chart):**
- **The x-axis** is the number of components kept.
- **The y-axis** is the total fraction of variance explained so far.
- **The red dashed line** marks the 95% threshold — a common target that means "keep enough components to retain 95% of the information."
- **The vertical dotted line** shows exactly how many components you need to hit 95%.

**What this tells you:**
- The MNIST dataset has 64 features, but most of the information is captured in far fewer components.
- You can dramatically shrink the feature space with almost no loss of information.

In [ ]:
# Scree plot — eigenvalues (variance per component, not ratio)
eigenvalues = pca_full.explained_variance_

plt.figure(figsize=(10, 5))
plt.plot(range(1, 31), eigenvalues[:30], color='steelblue', linewidth=2,
         marker='o', markersize=6, markerfacecolor='white', markeredgewidth=2)
plt.xlabel('Principal Component Number')
plt.ylabel('Eigenvalue (Variance)')
plt.title('Scree Plot — PCA on MNIST Digits\nLook for the "elbow" to decide how many components to keep')
plt.axvline(n_95, color='tomato', linestyle='--', linewidth=1.5,
            label=f'Elbow region around component {n_95}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Eigenvalue of PC1: {eigenvalues[0]:.2f}")
print(f"Eigenvalue of PC{n_95}: {eigenvalues[n_95-1]:.2f}")
print(f"Eigenvalue of PC64: {eigenvalues[63]:.4f}")

### How to Read This Chart: Scree Plot

A **scree plot** shows the eigenvalue (raw variance, not percentage) of each principal component.

- **The x-axis** is the component number (1, 2, 3…).
- **The y-axis** is the eigenvalue — how much raw variance that component explains.
- **The line drops steeply at first**, then levels off. The point where it bends is called the **elbow**.

**The elbow method:**
- Components to the **left of the elbow** are capturing real signal — keep those.
- Components to the **right of the elbow** are capturing mostly noise — you can discard those.
- This is the exact same elbow intuition used in K-Means clustering (choosing the number of clusters by the elbow of the inertia curve).

**Practical rule:**
- The elbow method and the 95% cumulative variance rule often point to similar answers.
- If they disagree, prefer the 95% rule for precision, but trust the elbow for interpretability and simplicity.

---

## Section 4 — PCA 2D Visualization

We reduce MNIST to exactly 2 dimensions using PCA. This lets us plot every digit as a single dot in a 2D scatter plot, colored by its label.

In [ ]:
# Reduce to 2D with PCA
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f"PCA 2D shape: {X_pca_2d.shape}")
print(f"Variance explained by PC1: {pca_2d.explained_variance_ratio_[0]:.1%}")
print(f"Variance explained by PC2: {pca_2d.explained_variance_ratio_[1]:.1%}")
print(f"Total (2D): {sum(pca_2d.explained_variance_ratio_):.1%}")

# Scatter plot colored by digit class
plt.figure(figsize=(10, 7))
palette = sns.color_palette('tab10', 10)
for digit in range(10):
    mask = y == digit
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                c=[palette[digit]], label=str(digit), alpha=0.6, s=20, edgecolors='none')

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA 2D Projection of MNIST Digits\nEach color = one digit class (0–9)')
plt.legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

### How to Read This Chart: PCA 2D Projection

This is a **scatter plot** where each dot is one digit image, projected down from 64 dimensions to just 2.

- **Each dot's color** tells you which digit it actually is (0 through 9).
- **Dots that cluster together** = images that are similar in the original 64-dimensional space.
- **The x-axis (PC1)** represents the direction of maximum variance — the single direction that best separates the data.
- **The y-axis (PC2)** represents the second-most important direction.

**What you should notice:**
- Some digit classes partially separate — for example, digit 0 and digit 1 often end up in different regions.
- But many classes **overlap heavily** — PCA's 2D projection is blurry because it only captures ~28% of the total variance.
- **PCA preserves global structure** — digits that look very different from each other (e.g., 0 vs. 1) tend to land far apart. Digits that look similar (e.g., 4 and 9) overlap more.

**The key limitation of PCA for visualization:**
> PCA optimizes for variance, not for class separation. It does not know there are 10 classes — it just finds directions of spread. For cleaner cluster separation, t-SNE and UMAP do much better (as we will see).

---

## Section 5 — PCA as Preprocessing Before a Classifier

One of the most practical uses of PCA is **not visualization** — it is preprocessing before machine learning.

The idea: reduce the feature space before feeding data into a classifier. This can:
- Speed up training
- Remove noise dimensions that hurt generalization
- Act as a form of regularization

We compare three approaches on MNIST digits with Logistic Regression:
1. **Raw features** — all 64 pixel values
2. **PCA(95% variance)** — keep however many components explain 95% of variance
3. **PCA(20 components)** — fixed 20 components (aggressive compression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

results_preprocessing = []

# ── Approach 1: Raw features (64) ──────────────────────────────────
lr_raw = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
t0 = time.time()
lr_raw.fit(X_train, y_train)
t_raw = time.time() - t0
acc_raw = accuracy_score(y_test, lr_raw.predict(X_test))
results_preprocessing.append({'Approach': 'Raw (64 features)', 'n_features': 64,
                               'Accuracy': acc_raw, 'Train Time (s)': round(t_raw, 3)})

# ── Approach 2: PCA(95% variance) ─────────────────────────────────
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca95 = pca_95.fit_transform(X_train)
X_test_pca95  = pca_95.transform(X_test)
n_components_95 = pca_95.n_components_

lr_pca95 = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
t0 = time.time()
lr_pca95.fit(X_train_pca95, y_train)
t_pca95 = time.time() - t0
acc_pca95 = accuracy_score(y_test, lr_pca95.predict(X_test_pca95))
results_preprocessing.append({'Approach': f'PCA(95%) → {n_components_95} features',
                               'n_features': n_components_95,
                               'Accuracy': acc_pca95, 'Train Time (s)': round(t_pca95, 3)})

# ── Approach 3: PCA(20 components) ─────────────────────────────────
pca_20 = PCA(n_components=20, random_state=RANDOM_STATE)
X_train_pca20 = pca_20.fit_transform(X_train)
X_test_pca20  = pca_20.transform(X_test)

lr_pca20 = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
t0 = time.time()
lr_pca20.fit(X_train_pca20, y_train)
t_pca20 = time.time() - t0
acc_pca20 = accuracy_score(y_test, lr_pca20.predict(X_test_pca20))
results_preprocessing.append({'Approach': 'PCA(20 components)',
                               'n_features': 20,
                               'Accuracy': acc_pca20, 'Train Time (s)': round(t_pca20, 3)})

# Display results table
df_preprocessing = pd.DataFrame(results_preprocessing)
df_preprocessing['Accuracy'] = df_preprocessing['Accuracy'].map('{:.4f}'.format)
print("PCA as Preprocessing — Comparison Table")
print(df_preprocessing.to_string(index=False))

### How to Read This Table: PCA as Preprocessing

This table compares three versions of the same classifier (Logistic Regression) with different levels of PCA compression applied to the input features.

| Column | What It Means |
|--------|---------------|
| **Approach** | Which version: raw, PCA at 95%, or PCA with fixed 20 components |
| **n_features** | How many features (dimensions) the classifier actually sees |
| **Accuracy** | Fraction of test digits correctly classified |
| **Train Time (s)** | How long the classifier took to train |

**What to look for:**
- PCA at 95% variance keeps most of the accuracy with significantly fewer features.
- PCA with 20 components (aggressive compression) shows whether performance degrades when you push compression hard.
- Training time should decrease with fewer features — PCA can make classifiers faster.

**Why PCA sometimes improves accuracy:**
> PCA removes low-variance dimensions, which are often noise. By discarding noise, the classifier sees a cleaner signal — similar to how regularization works. This can improve generalization even when accuracy on this test set looks similar.

**The practical lesson:**
> If you can maintain accuracy while cutting features from 64 to 20, you get a 3x smaller model that trains faster, uses less memory, and generalizes better.

---

## Section 6 — Incremental PCA (for Large Datasets)

### What Is the Problem with Regular PCA on Large Data?

Standard PCA needs to load the **entire dataset into memory at once** to compute its covariance matrix. If you have 10 million images, that is simply not feasible.

**IncrementalPCA** solves this by processing data in **mini-batches**:
1. Load batch 1 → update the running estimate of the covariance matrix
2. Load batch 2 → update again
3. Repeat until all data has been processed

The result is mathematically identical (or nearly identical) to regular PCA, but you only ever need one batch in RAM at a time.

### When to Use IncrementalPCA
- Dataset does **not fit in RAM** (e.g., large image datasets, genomic data)
- **Streaming data** — when new data arrives continuously and you want to update the PCA model
- When training a PCA model on a **cluster** where each node holds only a shard of the data

In [ ]:
# Incremental PCA — process data in batches of 200 samples
BATCH_SIZE = 200
N_COMPONENTS = 20

ipca = IncrementalPCA(n_components=N_COMPONENTS)

# Feed data in chunks — simulating a large dataset that won't fit in RAM
for batch_start in range(0, len(X_scaled), BATCH_SIZE):
    batch = X_scaled[batch_start : batch_start + BATCH_SIZE]
    if len(batch) >= N_COMPONENTS:  # IncrementalPCA needs batch_size >= n_components
        ipca.partial_fit(batch)

# Transform using IncrementalPCA
X_ipca = ipca.transform(X_scaled)

# Regular PCA with same n_components for comparison
pca_compare = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca_compare = pca_compare.fit_transform(X_scaled)

# Compare explained variance: should be very close
print("Comparing IncrementalPCA vs regular PCA (20 components)")
print(f"{'Component':<12} {'PCA Var':>12} {'IPCA Var':>12}")
print("-" * 38)
for i in range(5):
    print(f"PC{i+1:<10} {pca_compare.explained_variance_ratio_[i]:>12.4f} "
          f"{ipca.explained_variance_ratio_[i]:>12.4f}")

total_pca  = pca_compare.explained_variance_ratio_.sum()
total_ipca = ipca.explained_variance_ratio_.sum()
print(f"\nTotal (20 components): PCA = {total_pca:.4f} | IncrementalPCA = {total_ipca:.4f}")
print("\nConclusion: IncrementalPCA produces nearly identical results to regular PCA.")
print("The tiny differences come from numerical precision in the batch-wise updates.")

### When to Use IncrementalPCA: A Quick Summary

| Situation | Use Regular PCA | Use IncrementalPCA |
|-----------|----------------|--------------------|
| Data fits in RAM | ✅ Preferred (faster) | Works, but unnecessary |
| Data does NOT fit in RAM | ❌ Will crash | ✅ Required |
| Streaming / online learning | ❌ Needs full data | ✅ Ideal |
| Need exact results | ✅ Exact | ✅ Near-identical |
| Very large batch possible | — | ✅ Larger batches = more accurate |

**Rule of thumb:** The larger your batches, the closer IncrementalPCA gets to regular PCA. When `batch_size == n_samples`, they are identical. Use the smallest batch size that fits in your RAM.

---

## Section 7 — t-SNE

**t-SNE converts pairwise distances into probabilities and tries to preserve neighborhood structure in 2D.**

Here is the plain-English walkthrough:

1. **In the original high-D space:** For each point, compute how likely each other point is to be its "neighbor" — closer points get higher probability. This uses a Gaussian distribution centered at each point.
2. **In the 2D output space:** Start with a random layout. Assign neighbor-probabilities again, but now using a Student's t-distribution (heavier tails than Gaussian).
3. **Optimize:** Shuffle the 2D points until the two probability distributions match as closely as possible.

**Why the t-distribution?**  
It has heavier tails — meaning it can put points that are "sort of" neighbors far apart in 2D. This creates the clean, well-separated clusters t-SNE is famous for.

### Perplexity — The Key Hyperparameter

**Perplexity** controls how many nearby neighbors each point "cares about."

> Think of it as the number of effective neighbors. Perplexity = 5 means each point focuses on its 5 closest neighbors. Perplexity = 50 means each point considers a much wider neighborhood.

- **Low perplexity (e.g., 5):** Very local view — clusters may fragment into many tiny subclusters
- **High perplexity (e.g., 50):** More global view — clusters may blur together
- **Sweet spot (typically 20–50):** The sklearn default is 30, which works well in most cases

**Always run t-SNE at multiple perplexity values and compare.** A result that looks the same across different perplexities is more trustworthy.

In [ ]:
# t-SNE perplexity sensitivity — run at 4 different values
# Note: t-SNE is slow — we run on a subset of 800 samples for speed
np.random.seed(RANDOM_STATE)
subset_idx = np.random.choice(len(X_scaled), size=800, replace=False)
X_sub = X_scaled[subset_idx]
y_sub = y[subset_idx]

perplexities = [5, 15, 30, 50]
tsne_results_perp = {}

for perp in perplexities:
    print(f"Running t-SNE with perplexity={perp}...", end=' ')
    t0 = time.time()
    tsne = TSNE(n_components=2, perplexity=perp, random_state=RANDOM_STATE,
                n_iter=1000, learning_rate='auto', init='pca')
    X_tsne = tsne.fit_transform(X_sub)
    tsne_results_perp[perp] = X_tsne
    print(f"done in {time.time()-t0:.1f}s")

# 4-panel subplot
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
palette = sns.color_palette('tab10', 10)

for ax, perp in zip(axes, perplexities):
    X_tsne = tsne_results_perp[perp]
    for digit in range(10):
        mask = y_sub == digit
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                   c=[palette[digit]], s=15, alpha=0.7, edgecolors='none')
    ax.set_title(f'Perplexity = {perp}', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('t-SNE Perplexity Sensitivity — MNIST Digits (n=800)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: t-SNE Perplexity Effect

Four side-by-side scatter plots of the **same 800 digit images**, projected to 2D with t-SNE at different perplexity values. Each color is one digit class.

**Panel 1 — Perplexity = 5 (very local):**
- Each cluster tends to **fragment** into many small isolated subclusters.
- At this setting, t-SNE only cares about each point's 5 nearest neighbors — so it creates tight micro-clusters but does not see the bigger picture.
- The layout looks "spikey" or "frayed."

**Panel 2 — Perplexity = 15:**
- Clusters start to consolidate. The structure becomes clearer.
- Still somewhat fragmented for some digits, but major groups are emerging.

**Panel 3 — Perplexity = 30 (the standard default):**
- Clean, well-separated clusters for most digits.
- This is typically the sweet spot — local neighborhoods are well-preserved AND the overall layout makes sense.
- Most t-SNE papers and tutorials use perplexity=30 as the default starting point.

**Panel 4 — Perplexity = 50 (very global):**
- Clusters may start to compress or merge slightly.
- The layout looks more "smeared" — individual clusters may not be as crisp.

**Practical rule:**
> Perplexity = 30 is a safe default. Always check perplexity = 5 and perplexity = 50 to make sure your main conclusions hold across settings. If the cluster structure you see only appears at one specific perplexity, be skeptical.

---

## Section 8 — t-SNE 2D Visualization

Now we run t-SNE with the standard `perplexity=30` on a larger subset for a clean, publication-quality visualization.

In [ ]:
# Run t-SNE(perplexity=30) on full dataset
print("Running t-SNE (perplexity=30) on full MNIST digits dataset...")
t0 = time.time()
tsne_30 = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE,
               n_iter=1000, learning_rate='auto', init='pca')
X_tsne_2d = tsne_30.fit_transform(X_scaled)
tsne_time = time.time() - t0
print(f"t-SNE completed in {tsne_time:.1f}s")

# Scatter plot
plt.figure(figsize=(10, 7))
palette = sns.color_palette('tab10', 10)
for digit in range(10):
    mask = y == digit
    plt.scatter(X_tsne_2d[mask, 0], X_tsne_2d[mask, 1],
                c=[palette[digit]], label=str(digit),
                alpha=0.6, s=20, edgecolors='none')

plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.title('t-SNE 2D Projection of MNIST Digits (perplexity=30)\nEach color = one digit class (0–9)')
plt.legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

### How to Read This Chart: t-SNE 2D Projection

Same scatter plot format as PCA — each dot is one digit image, colored by class — but the result looks very different.

**What you should immediately notice compared to PCA:**
- **Much cleaner separation** — most digit classes form tight, distinct clusters with minimal overlap.
- Digit 0 (purple), digit 1 (blue), digit 6 (brown) often form very clean clusters.
- Digit 4 and digit 9 may still overlap slightly because their shapes are similar.

**A critical warning about t-SNE interpretation:**
- The **distances between clusters do NOT mean anything** in t-SNE. If cluster 0 is far from cluster 1, that does not mean digit 0 and digit 1 are more different than, say, clusters that are closer. t-SNE only preserves **local** structure — which points are neighbors — not **global** distances between groups.
- **Cluster sizes do not mean anything** either — a large cluster does not indicate more samples or more variance.
- The overall layout (which side of the plot a cluster lands on) changes with different random seeds and perplexity settings.

---

> ⚠️ **WARNING: t-SNE is for VISUALIZATION ONLY.**
> 
> **Never use t-SNE embeddings as features for a classifier.** The 2D coordinates produced by t-SNE are not meaningful representations of the data — they are artifacts of a non-linear optimization process tuned for visual separation. Using them as features for downstream ML will give misleading results. For features, use PCA or UMAP instead.

---

## Section 9 — UMAP

**UMAP preserves both local structure (like t-SNE) AND global structure (like PCA), and is much faster.**

UMAP (Uniform Manifold Approximation and Projection) is based on Riemannian geometry and algebraic topology — but you do not need to know the math. The key ideas in plain English:

1. **Build a graph in high-D space** — connect each point to its k nearest neighbors. The edge weights reflect how similar the points are.
2. **Build a matching graph in 2D** — starting from a random layout, adjust points to make the 2D graph match the high-D graph as closely as possible.
3. **Optimization is fast** — UMAP uses stochastic gradient descent with clever sampling, making it dramatically faster than t-SNE.

### UMAP vs t-SNE vs PCA at a Glance

| Property | PCA | t-SNE | UMAP |
|----------|-----|-------|------|
| Preserves global structure | ✅ Yes | ❌ No | ✅ Yes |
| Preserves local structure | ⚠️ Partial | ✅ Yes | ✅ Yes |
| Speed | ✅ Very fast | ❌ Slow | ✅ Fast |
| Can use as ML features | ✅ Yes | ❌ No | ✅ Yes |
| Handles large datasets | ✅ With IncrementalPCA | ❌ Very slow | ✅ Yes |
| Cluster distances meaningful | ✅ Partial | ❌ No | ✅ Yes |

In [ ]:
# UMAP 2D projection
if UMAP_AVAILABLE:
    print("Running UMAP (n_components=2)...")
    t0 = time.time()
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
    X_umap_2d = reducer.fit_transform(X_scaled)
    umap_time = time.time() - t0
    print(f"UMAP completed in {umap_time:.1f}s")

    # Scatter plot
    plt.figure(figsize=(10, 7))
    palette = sns.color_palette('tab10', 10)
    for digit in range(10):
        mask = y == digit
        plt.scatter(X_umap_2d[mask, 0], X_umap_2d[mask, 1],
                    c=[palette[digit]], label=str(digit),
                    alpha=0.6, s=20, edgecolors='none')

    plt.xlabel('UMAP Dimension 1')
    plt.ylabel('UMAP Dimension 2')
    plt.title('UMAP 2D Projection of MNIST Digits\nEach color = one digit class (0–9)')
    plt.legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=2)
    plt.tight_layout()
    plt.show()
else:
    print("UMAP not available. Install with: pip install umap-learn")
    # Fallback: use PCA as a placeholder for comparisons below
    X_umap_2d = X_pca_2d.copy()
    umap_time = 0.0

In [ ]:
# Side-by-side comparison: PCA vs t-SNE vs UMAP
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
palette = sns.color_palette('tab10', 10)

datasets = [
    (X_pca_2d,  'PCA 2D',  'Global structure preserved, blurry clusters'),
    (X_tsne_2d, 't-SNE 2D', 'Tight clusters, cluster distances meaningless'),
    (X_umap_2d, 'UMAP 2D',  'Tight clusters AND relative positions meaningful'),
]

for ax, (X_embed, title, subtitle) in zip(axes, datasets):
    for digit in range(10):
        mask = y == digit
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1],
                   c=[palette[digit]], label=str(digit),
                   alpha=0.6, s=15, edgecolors='none')
    ax.set_title(f'{title}\n{subtitle}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# Shared legend
handles = [plt.scatter([], [], c=[palette[d]], s=40, label=str(d)) for d in range(10)]
fig.legend(handles=handles, title='Digit', loc='lower center',
           ncol=10, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('PCA vs t-SNE vs UMAP — Same Data, Same Colors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: PCA vs t-SNE vs UMAP

Three scatter plots of the **same 1,797 digit images** projected to 2D by three different methods.

**Left — PCA:**
- Clusters are **blurry and overlapping**. Many colors mix together.
- The overall layout has global meaning — points that are far apart in PCA space are genuinely different.
- But local structure (tight clusters per class) is poor.

**Middle — t-SNE:**
- **Tight, well-separated clusters** — each digit class forms its own island.
- The separation is dramatically better than PCA.
- BUT: the distances **between** clusters are arbitrary. Whether cluster 0 is near cluster 1 or far from cluster 1 in this plot conveys no information.

**Right — UMAP:**
- Also produces **tight clusters** similar to t-SNE.
- Additionally, the **relative positions of clusters are meaningful** — if cluster 4 is close to cluster 9 in the UMAP layout, they really are more similar to each other than to cluster 0.
- UMAP preserves both local AND global structure.

**The one-line summary:**
> PCA = global structure, blurry clusters. t-SNE = tight clusters, no global meaning. UMAP = tight clusters AND relative cluster positions are meaningful.

---

## Section 10 — UMAP vs t-SNE: Speed Benchmark

One of UMAP's most practical advantages is **speed**. t-SNE has O(n²) complexity — doubling the number of samples quadruples the computation time. UMAP scales much more gracefully.

Let us measure the actual difference on our dataset.

In [ ]:
# Speed benchmark on full dataset (or subset if dataset is large)
# We already have tsne_time and umap_time from above — report them cleanly

if UMAP_AVAILABLE and umap_time > 0:
    speedup = tsne_time / umap_time if umap_time > 0 else float('inf')
    print("Speed Benchmark — MNIST Digits (n=1797, 64 features → 2D)")
    print("-" * 55)
    print(f"  t-SNE time  : {tsne_time:.2f}s")
    print(f"  UMAP time   : {umap_time:.2f}s")
    print(f"  Speedup     : {speedup:.1f}x faster with UMAP")
    print()
    print("Note: on larger datasets (n > 10,000), the difference is even more dramatic.")
    print("UMAP is typically 10–30x faster than t-SNE on large datasets.")
    print("For datasets > 10,000 samples, UMAP is the practical choice.")
else:
    print(f"t-SNE time : {tsne_time:.2f}s")
    print("UMAP not available for comparison.")
    print("\nTypical benchmark (n=10,000 samples):")
    print("  t-SNE : ~120–300 seconds")
    print("  UMAP  : ~10–20 seconds")
    print("  Speedup: typically 10–30x")

# Bar chart comparing times
methods = ['t-SNE', 'UMAP'] if UMAP_AVAILABLE else ['t-SNE']
times_list = [tsne_time, umap_time] if UMAP_AVAILABLE else [tsne_time]

plt.figure(figsize=(6, 4))
bars = plt.bar(methods, times_list, color=['tomato', 'steelblue'][:len(methods)],
               edgecolor='navy', alpha=0.8, width=0.4)
for bar, t in zip(bars, times_list):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{t:.1f}s', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylabel('Time (seconds)')
plt.title('Runtime Comparison: t-SNE vs UMAP\n(MNIST digits, n=1797)')
plt.tight_layout()
plt.show()

### Speed Summary

| Dataset Size | t-SNE (approx.) | UMAP (approx.) | Speedup |
|-------------|-----------------|----------------|----------|
| 1,797 (this notebook) | ~10–30s | ~1–5s | ~5–10x |
| 10,000 samples | ~2–5 min | ~5–15s | ~10–20x |
| 100,000 samples | ~hours | ~1–5 min | ~30–60x |

**UMAP is typically 10–30x faster than t-SNE on large datasets. For datasets > 10,000 samples, UMAP is the practical choice.**

Additionally, unlike t-SNE, UMAP can:
- Transform **new data points** (fit on training data, apply to test data)
- Reduce to **more than 2 dimensions** for use as ML features
- Be used in production pipelines without re-running the full algorithm on every new point

---

## Section 11 — Autoencoder for Dimensionality Reduction

**An Autoencoder learns a compressed representation by training a neural network to reconstruct its input.**

Unlike PCA (linear), t-SNE (for visualization only), or UMAP (graph-based), an Autoencoder is a **neural network** — it can learn highly non-linear compressions.

### Architecture — The Encoder-Bottleneck-Decoder Pattern

```
Input (64)  →  Encoder  →  Bottleneck (2)  →  Decoder  →  Reconstructed Input (64)
  [64]         [64→32→16→8→2]     [2]         [2→8→16→32→64]          [64]
```

- **Encoder** — compresses the input, layer by layer, into a small latent vector
- **Bottleneck** — the smallest layer — the compressed representation (latent space)
- **Decoder** — expands the latent vector back to the original size
- **Loss** — mean squared error between input and reconstruction

The network learns to keep only the most important information in the bottleneck — everything else gets lost in reconstruction. This forces the bottleneck to be a good, information-dense summary of the input.

### Why Use an Autoencoder vs PCA?
- Autoencoder can learn **non-linear** relationships — PCA cannot
- Autoencoder can model complex structure that PCA misses
- Trade-off: Autoencoder needs more data, more training time, and more tuning

In [ ]:
# ── PyTorch Autoencoder ────────────────────────────────────────────

class Autoencoder(nn.Module):
    """Simple fully-connected autoencoder with 2D bottleneck."""

    def __init__(self):
        super(Autoencoder, self).__init__()

        # Encoder: 64 → 32 → 16 → 8 → 2
        self.encoder = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)   # Bottleneck — 2D latent space
        )

        # Decoder: 2 → 8 → 16 → 32 → 64
        self.decoder = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.Sigmoid()       # Output in [0,1] range
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

    def encode(self, x):
        return self.encoder(x)


# Normalize to [0,1] for Sigmoid output activation
X_norm = X / X.max()   # MNIST pixel range: 0–16, normalize to 0–1

# Build DataLoader
X_tensor = torch.FloatTensor(X_norm)
dataset   = TensorDataset(X_tensor)
loader    = DataLoader(dataset, batch_size=64, shuffle=True)

# Instantiate model and optimizer
model     = Autoencoder()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Train for 50 epochs
EPOCHS = 50
train_losses = []

print(f"Training Autoencoder for {EPOCHS} epochs...")
model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for (batch_x,) in loader:
        optimizer.zero_grad()
        output = model(batch_x)
        loss   = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_x)
    epoch_loss /= len(X_norm)
    train_losses.append(epoch_loss)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS}  |  Loss: {epoch_loss:.6f}")

print("Training complete.")

In [ ]:
# Plot training loss curve
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, color='steelblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss (MSE)')
plt.title('Autoencoder Training Loss Over 50 Epochs\nLoss should decrease smoothly toward a plateau')
plt.tight_layout()
plt.show()

### How to Read This Chart: Autoencoder Training Loss

This is a standard **training loss curve** — one of the most fundamental charts in deep learning.

- **The x-axis** is the epoch number (one pass through the entire training dataset).
- **The y-axis** is the mean squared error between each image and its reconstruction.
- **A healthy loss curve** starts high and drops sharply in the first few epochs, then continues to decrease more slowly, eventually leveling off (plateauing).

**What to look for:**
- **Smooth, monotonically decreasing** — the autoencoder is learning steadily. Good sign.
- **Sudden spikes** — the learning rate may be too high, or there is instability in training.
- **Plateau very early (after epoch 5–10)** — the bottleneck is too small. The network has hit the limits of what 2 dimensions can represent. Try a larger bottleneck (e.g., 8 or 16) if you need better reconstruction.
- **Still decreasing at epoch 50** — train for more epochs; the model hasn't converged yet.

**Rule of thumb:**
> If the loss plateaus early, you have two options: increase the bottleneck size (better reconstruction, less compression) or train longer with a lower learning rate. For 2D visualization purposes, some blurriness is expected and acceptable.

In [ ]:
# Compare original vs reconstructed images
model.eval()
with torch.no_grad():
    X_reconstructed = model(X_tensor).numpy()

# Pick 8 random samples
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_norm), size=8, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for col, idx in enumerate(sample_idx):
    # Original
    axes[0, col].imshow(X_norm[idx].reshape(8, 8), cmap='gray_r', vmin=0, vmax=1)
    axes[0, col].set_title(f'Label: {y[idx]}', fontsize=8)
    axes[0, col].axis('off')
    # Reconstructed
    axes[1, col].imshow(X_reconstructed[idx].reshape(8, 8), cmap='gray_r', vmin=0, vmax=1)
    axes[1, col].set_title('Recon.', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', rotation=90, fontsize=10, labelpad=30)
axes[1, 0].set_ylabel('Reconstructed', rotation=90, fontsize=10, labelpad=5)

plt.suptitle('Autoencoder: Original vs Reconstructed Digits (2D Bottleneck)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Reconstruction Quality

Two rows of digit images — **top row = originals, bottom row = reconstructions** from the 2D bottleneck.

- The autoencoder compressed each 64-pixel image to just **2 numbers** (the bottleneck), then expanded it back to 64 pixels.
- You should expect the reconstructed images to look **blurry or simplified** — the 2D bottleneck is a very aggressive compression.
- The fact that the digit shapes are still recognizable shows that the autoencoder learned a meaningful 2D representation.

**Interpreting reconstruction quality:**
- **Blurry but correct shape** — the 2D bottleneck captures the essential structure, but loses detail. This is expected for 64→2.
- **Reconstructions look like wrong digits** — the bottleneck is too small for the complexity of the data.
- **Reconstructions look almost perfect** — the bottleneck is large enough to capture everything. At 2D, this would be surprising; at 20D, it is more expected.

**The compression trade-off:**
> Perfect reconstruction = no compression (bottleneck too large to force the network to learn a useful summary)  
> Very blurry reconstruction = too compressed (bottleneck too small to capture enough information)  
> The right bottleneck size depends on your task — for visualization, 2D is useful; for compression, 8–32 dimensions give much better quality.

In [ ]:
# Extract 2D latent space and plot scatter
model.eval()
with torch.no_grad():
    X_autoenc_2d = model.encode(X_tensor).numpy()

plt.figure(figsize=(10, 7))
palette = sns.color_palette('tab10', 10)
for digit in range(10):
    mask = y == digit
    plt.scatter(X_autoenc_2d[mask, 0], X_autoenc_2d[mask, 1],
                c=[palette[digit]], label=str(digit),
                alpha=0.6, s=20, edgecolors='none')

plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Autoencoder 2D Latent Space — MNIST Digits\nEach color = one digit class (0–9)')
plt.legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

print("Latent space shape:", X_autoenc_2d.shape)
print("Comparing to PCA and UMAP: the autoencoder latent space may show different cluster structure")
print("because it is optimized for reconstruction, not for class separation.")

---

## Section 12 — Final Comparison: All 4 Methods

Now we put everything side by side — the same dataset, the same colors, four different methods. This is the definitive comparison.

In [ ]:
# 4-panel comparison: PCA, t-SNE, UMAP, Autoencoder
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
palette = sns.color_palette('tab10', 10)

all_methods = [
    (X_pca_2d,      'PCA 2D',         'Linear | Preserves global'),
    (X_tsne_2d,     't-SNE 2D',       'Non-linear | Local only'),
    (X_umap_2d,     'UMAP 2D',        'Non-linear | Local + Global'),
    (X_autoenc_2d,  'Autoencoder 2D', 'Neural Net | Reconstruction-based'),
]

for ax, (X_embed, title, subtitle) in zip(axes, all_methods):
    for digit in range(10):
        mask = y == digit
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1],
                   c=[palette[digit]], label=str(digit),
                   alpha=0.6, s=15, edgecolors='none')
    ax.set_title(f'{title}\n{subtitle}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# Shared legend at the bottom
handles = [plt.scatter([], [], c=[palette[d]], s=50, label=str(d)) for d in range(10)]
fig.legend(handles=handles, title='Digit', loc='lower center',
           ncol=10, bbox_to_anchor=(0.5, -0.08))

plt.suptitle('All 4 Dimensionality Reduction Methods — Same Data, Same Colors',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Time comparison table
print("Computing fit_transform times...")

# PCA time
t0 = time.time()
_ = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
pca_time = time.time() - t0

# t-SNE time (already measured)
# UMAP time (already measured)

# Autoencoder encode time (inference only, training excluded)
t0 = time.time()
with torch.no_grad():
    _ = model.encode(X_tensor).numpy()
ae_infer_time = time.time() - t0

# Build comparison table
time_data = [
    {'Method': 'PCA',          'fit_transform time (s)': round(pca_time, 3)},
    {'Method': 't-SNE',        'fit_transform time (s)': round(tsne_time, 3)},
    {'Method': 'UMAP',         'fit_transform time (s)': round(umap_time, 3) if UMAP_AVAILABLE else 'N/A'},
    {'Method': 'Autoencoder',  'fit_transform time (s)': round(ae_infer_time, 4)
                                                          if ae_infer_time < 1 else round(ae_infer_time, 3)},
]
df_times = pd.DataFrame(time_data)
print("\nRuntime Comparison (fit_transform on n=1797 samples, 64 features → 2D)")
print(df_times.to_string(index=False))
print("\nNote: Autoencoder time is inference only (not including 50 epochs of training).")

In [ ]:
# Silhouette scores — quantify how well-separated the 2D clusters are
print("Computing silhouette scores (higher = better class separation in 2D)...")
print("(Subsetting to 500 samples for speed)")
np.random.seed(RANDOM_STATE)
sil_idx = np.random.choice(len(y), size=500, replace=False)

sil_results = []
for embed, name in [
    (X_pca_2d,     'PCA'),
    (X_tsne_2d,    't-SNE'),
    (X_umap_2d,    'UMAP'),
    (X_autoenc_2d, 'Autoencoder'),
]:
    score = silhouette_score(embed[sil_idx], y[sil_idx])
    sil_results.append({'Method': name, 'Silhouette Score': round(score, 4)})
    print(f"  {name:<15} silhouette = {score:.4f}")

df_sil = pd.DataFrame(sil_results).sort_values('Silhouette Score', ascending=False)
print("\nRanking (highest silhouette = clearest cluster separation):")
print(df_sil.to_string(index=False))

### How to Read This Chart: Which Method to Use?

This **4-panel comparison** is the most important chart in the notebook — it shows all four methods on the same data.

**Silhouette Score** measures how well-separated the digit clusters are in 2D:
- Score close to **+1.0** = very well-separated clusters (each point is much closer to its own class than any other class)
- Score close to **0.0** = overlapping clusters
- Score below **0** = points are in the wrong cluster

**Visual interpretation:**

| Method | Cluster Separation | Distances Meaningful | Speed | Use as ML Features |
|--------|-------------------|--------------------|-------|-------------------|
| **PCA** | Moderate (overlapping) | ✅ Yes | ✅ Fastest | ✅ Yes |
| **t-SNE** | Best visually | ❌ No | ❌ Slow | ❌ No |
| **UMAP** | Excellent | ✅ Yes | ✅ Fast | ✅ Yes |
| **Autoencoder** | Good (varies with training) | ⚠️ Partial | ⚠️ Slow to train | ✅ Yes |

**When to use which:**
> - **Quick exploratory visualization on any dataset?** → t-SNE with perplexity=30
> - **Need to visualize AND compare cluster sizes/positions?** → UMAP
> - **Preprocessing before a classifier?** → PCA or UMAP
> - **Compression with reconstruction ability?** → Autoencoder
> - **Dataset > 10K samples?** → UMAP or IncrementalPCA (avoid t-SNE)

---

## Section 13 — Summary and Key Takeaways

### The Complete Comparison Table

| Method | Preserves Global Structure | Preserves Local Structure | Speed | Can Use as ML Features | Interpretable | Best For |
|--------|--------------------------|--------------------------|-------|----------------------|--------------|----------|
| **PCA** | ✅ Yes | ⚠️ Partial | ✅ Very fast | ✅ Yes | ⚠️ Partially | Preprocessing, compression, noise removal |
| **IncrementalPCA** | ✅ Yes | ⚠️ Partial | ✅ Very fast | ✅ Yes | ⚠️ Partially | Large datasets that don't fit in RAM |
| **t-SNE** | ❌ No | ✅ Yes | ❌ Slow | ❌ No | ❌ No | Visualization only |
| **UMAP** | ✅ Yes | ✅ Yes | ✅ Fast | ✅ Yes | ❌ No | Visualization + features + large data |
| **Autoencoder** | ⚠️ Depends | ✅ Yes | ⚠️ Slow to train | ✅ Yes | ❌ No | Non-linear compression + reconstruction |

---

### Decision Guide

Use this flowchart to choose the right method:

```
Need features for a downstream ML model?
  → Yes: Use PCA or UMAP
        → Dataset is huge (>100K samples)? → UMAP or IncrementalPCA
        → Need reconstruction (e.g., compression, denoising)? → Autoencoder
        → Simple, interpretable features needed? → PCA
  → No (visualization only):
        → Want the cleanest cluster separation? → t-SNE (perplexity=30)
        → Want cluster distances to be meaningful? → UMAP
        → Dataset > 10K samples? → UMAP (t-SNE too slow)
```

---

### Critical Warnings — Common Mistakes

1. **t-SNE for ML features = WRONG**  
   The 2D coordinates from t-SNE are non-deterministic artifacts of an optimization process. They are not a stable, meaningful representation of the data. Training a classifier on t-SNE features will give misleading results.

2. **Distances in t-SNE = MEANINGLESS**  
   The fact that cluster A is close to cluster B in a t-SNE plot does not mean those groups are more similar. t-SNE only preserves local neighborhood structure, not global distances.

3. **PCA without scaling first = WRONG**  
   If one feature ranges from 0–1 and another from 0–1,000,000, PCA will be dominated by the large-scale feature. Always `StandardScaler()` before PCA.

4. **Assuming UMAP cluster distances are fully meaningful**  
   UMAP preserves global structure better than t-SNE, but the distances are still not as interpretable as PCA distances. Treat UMAP cluster positions as meaningful but not perfectly calibrated.

5. **Choosing bottleneck size without thought**  
   An autoencoder bottleneck of size 2 is for visualization. For compression or features, use 8–64 dimensions. Check reconstruction quality to find the right size.

---

### What's Next?

- Explore **clustering algorithms** in the companion notebook: `03_clustering.ipynb`
- Try **UMAP for supervised dimensionality reduction** — UMAP accepts class labels to improve cluster separation
- Explore **Variational Autoencoders (VAEs)** — a probabilistic extension of autoencoders that can generate new data points, not just compress existing ones
- Try **Kernel PCA** for non-linear dimensionality reduction using the kernel trick, without a neural network